In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats

In [2]:
df = pd.read_csv("../data/games_steam.csv")

### Dataset Selection

This dataset has been created with Steam store page scraping and Steam Games Scraper repository, which gathers data using the Steam API and Steam Spy.

This dataset is up to date as of March 2025, and contains two data versions - raw parsed dataset (games_march2025_full.csv), as well as a cleaned version without duplicate & playtest versions of games (games_march2025_cleaned.csv).

The cleaned version of the Steam Games Dataset was selected for this assessment. This version excludes duplicate entries and playtest versions of games improving data consistency.

Since the main objective of this assessment is to apply descriptive and inferential statistical techniques, the cleaned dataset allows the analysis to focus on statistical interpretation rather than extensive data preprocessing. However, further data quality checks will be performed before conducting the statistical analysis.

This dataset dates from March 2025

Dataset source: https://www.kaggle.com/datasets/artermiloff/steam-games-dataset?resource=download

License
MIT

### Dataset objectives

The objectives of this project is to analyse the Online game market, the context is, they just release one of the most expected games of the history (GTA VI), also, I have a background as a small game producer freelancer, I want to work in a dataset which make sense for my background. This will also help to select the best features and best analysis

In [3]:
df.head()

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,...,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,...,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,...,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,...,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517
4,359550,Tom Clancy's Rainbow Six® Siege,2015-12-01,17,3.99,9,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",NaN,...,682,2434,306,80,89916,"{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",84,1168020,76,12608


In [4]:
df.columns.tolist()

['appid',
 'name',
 'release_date',
 'required_age',
 'price',
 'dlc_count',
 'detailed_description',
 'about_the_game',
 'short_description',
 'reviews',
 'header_image',
 'website',
 'support_url',
 'support_email',
 'windows',
 'mac',
 'linux',
 'metacritic_score',
 'metacritic_url',
 'achievements',
 'recommendations',
 'notes',
 'supported_languages',
 'full_audio_languages',
 'packages',
 'developers',
 'publishers',
 'categories',
 'genres',
 'screenshots',
 'movies',
 'user_score',
 'score_rank',
 'positive',
 'negative',
 'estimated_owners',
 'average_playtime_forever',
 'average_playtime_2weeks',
 'median_playtime_forever',
 'median_playtime_2weeks',
 'discount',
 'peak_ccu',
 'tags',
 'pct_pos_total',
 'num_reviews_total',
 'pct_pos_recent',
 'num_reviews_recent']

In [5]:
df.shape

(89618, 47)

# Original dataset shape and Features selection

I openned the dataset in a sheet to analyse each column and verify the ones which are necessary to our analysis, i will call than "columns_to_keep", thats the first process of our data preparation

The original dataset has 89618 observations and 47 columns, we will now do some data preparation aiming to reduct this for better scope review

In [6]:
columns_to_keep = [
    "appid",
    "name",
    "release_date",
    "price",
    "mac",
    "categories",
    "genres",
    "metacritic_score",
    "positive",
    "negative",
    "average_playtime_forever",
    "median_playtime_forever",
    "peak_ccu",
    "pct_pos_total",
    "num_reviews_total"
]

df = df[columns_to_keep].copy()

df.shape

(89618, 15)

In [7]:
df.head()

,appid,name,release_date,price,mac,categories,genres,metacritic_score,positive,negative,average_playtime_forever,median_playtime_forever,peak_ccu,pct_pos_total,num_reviews_total
0,730,Counter-Strike 2,2012-08-21,0.00,False,"['Multi-player', 'Cross-Platform Multiplayer',...","['Action', 'Free To Play']",0,7480813,1135108,33189,5174,1212356,86,8632939
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0.00,False,"['Multi-player', 'PvP', 'Online PvP', 'Stats',...","['Action', 'Adventure', 'Massively Multiplayer...",0,1487960,1024436,0,0,616738,59,2513842
2,570,Dota 2,2013-07-09,0.00,True,"['Multi-player', 'Co-op', 'Steam Trading Cards...","['Action', 'Strategy', 'Free To Play']",90,1998462,451338,43031,898,555977,81,2452595
3,271590,Grand Theft Auto V Legacy,2015-04-13,0.00,False,"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Action', 'Adventure']",96,1719950,250012,19323,7101,117698,87,1803832
4,359550,Tom Clancy's Rainbow Six® Siege,2015-12-01,3.99,False,"['Single-player', 'Multi-player', 'PvP', 'Onli...",['Action'],0,1152763,218446,14204,2434,89916,84,1168020


In [39]:
df.tail()

,appid,name,release_date,price,mac,categories,genres,metacritic_score,positive,negative,average_playtime_forever,median_playtime_forever,peak_ccu,pct_pos_total,num_reviews_total
89613,2115110,Outrun Them,2024-02-29,3.99,False,"['Single-player', 'Steam Achievements', 'Stats...","['Indie', 'Early Access']",0,2,0,0,0,0,NaN,NaN
89614,1174200,Lands of Pharaoh: Episode 1,2020-01-07,10.00,False,"['Single-player', 'Family Sharing']","['Action', 'Adventure', 'Indie', 'Strategy']",0,2,0,0,0,0,NaN,NaN
89615,1160190,Player One,2019-10-10,1.99,False,"['Single-player', 'Family Sharing']","['Indie', 'Early Access']",0,2,1,0,0,0,NaN,NaN
89616,3380340,DragonRoad,2025-01-25,29.99,False,"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Adventure', 'RPG', 'Simulation', 'Strategy']",0,2,0,0,0,0,NaN,NaN
89617,763730,Ragtag Adventurers,2018-03-01,9.99,False,"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Action', 'Indie']",0,8,4,0,0,0,NaN,NaN


# Features selection explanation

We decided to keep these columns to focus on our project scope analysis

Categorical Variables as name; price; categories; genres; mac compatibility will be used to analyse correlation with Numerical variables such as Average/median playime; num_reviews (positive/negative); pct_pos_total; peak_ccu; metacritic score 

We also have release_date working as Temporal variable in our dataset

The removed features can be useful, but not in our scope this time, so thats why we made a .copy dataset to work with

## Features Description

appid — Unique application identifier assigned by Steam.

mac — Indicates whether the game is compatible with macOS.

categories — Game functionalities, such as multiplayer, single-player and achievements.

genres — Game classifications, such as Action, RPG and Strategy.

metacritic_score — Professional critic rating on Metacritic, ranging from 0 to 100.

positive — Number of positive user reviews.

negative — Number of negative user reviews.

average_playtime_forever — Average lifetime playtime in minutes.

median_playtime_forever — Median lifetime playtime in minutes.

peak_ccu — Peak Concurrent Users: maximum number of simultaneous players during the recorded period.

pct_pos_total — Percentage of positive user reviews.

num_reviews_total — Total number of user reviews reported by Steam.

In [8]:
df.isnull().sum()

appid                       0
name                        0
release_date                0
price                       0
mac                         0
categories                  0
genres                      0
metacritic_score            0
positive                    0
negative                    0
average_playtime_forever    0
median_playtime_forever     0
peak_ccu                    0
pct_pos_total               0
num_reviews_total           0
dtype: int64

# Missing Values Analysis

The dataset contains no explicit missing values across the 15 selected columns. However, an initial inspection revealed that several numerical variables contain zero values.

These values require further investigation, as they may represent valid observations or unavailable information recorded as zero.

In [9]:
numerical_cols = df.select_dtypes(include="number")

zero_count = (numerical_cols == 0).sum()

zero_count

appid                           0
price                       14160
metacritic_score            86071
positive                    19425
negative                    30396
average_playtime_forever    81608
median_playtime_forever     81608
peak_ccu                    70698
pct_pos_total                  18
num_reviews_total               0
dtype: int64

In [10]:
zero_percentage = (numerical_cols == 0).mean() * 100

zero_percentage.round(2)

appid                        0.00
price                       15.80
metacritic_score            96.04
positive                    21.68
negative                    33.92
average_playtime_forever    91.06
median_playtime_forever     91.06
peak_ccu                    78.89
pct_pos_total                0.02
num_reviews_total            0.00
dtype: float64

In [11]:
zero_analysis = pd.DataFrame({
    "Zero Count": zero_count,
    "Zero Percentage": zero_percentage.round(2)
})

zero_analysis.sort_values("Zero Percentage", ascending=False)

,Zero Count,Zero Percentage
metacritic_score,86071,96.04
average_playtime_forever,81608,91.06
median_playtime_forever,81608,91.06
peak_ccu,70698,78.89
negative,30396,33.92
positive,19425,21.68
price,14160,15.80
pct_pos_total,18,0.02
appid,0,0.00
num_reviews_total,0,0.00


In [12]:
#Checking for possible duplicates

df["appid"].duplicated().sum()

np.int64(0)

## Missing values analysis - Data Quality

We noticed that metacritic score, average/median playtime and peak_ccu have many Zero count values, we consider it Data issue but will not affect our project. We will use Feature engineering on our Data treatment applied to achieve our expected results.

Since we know this issue, we do not need to remove this observations, its just need to work aware of this inconsistency

# Data Treatment

In this milestone we will check the consistency of the collected informations - and treat than if necessary

## Data type analysis: check if the Temporal variables are the right typeform

In [13]:
df.dtypes

appid                         int64
name                         object
release_date                 object
price                       float64
mac                            bool
categories                   object
genres                       object
metacritic_score              int64
positive                      int64
negative                      int64
average_playtime_forever      int64
median_playtime_forever       int64
peak_ccu                      int64
pct_pos_total                 int64
num_reviews_total             int64
dtype: object

In [14]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    format="mixed",
    errors="coerce"
)

In [15]:
df["release_date"].dt.year.head()

0    2012
1    2017
2    2013
3    2015
4    2015
Name: release_date, dtype: int32

In [16]:
df["release_date"].dtype

dtype('<M8[ns]')

# Data type preparation

We wanted to check if our Temporal variable on the dataset was in the correct type format to be used, since we figured that was not, we fixed it and tested. Now, when needed to use this variable, Jupyter notebook will read with no issues

# Data Consistency check

We want to check the consistency of our data, since we trust the source but, as said in the API documentation, they extract information for either Steam, SteamSpy and web scraping.

We cannot continue before checking if the informations in our dataset are consistent

Starting from reviews consistency analysis (basically positive + negative should equals to num_reviews_total

In [17]:
review_consistency = (
    df["positive"] + df["negative"] == df["num_reviews_total"]
)

review_consistency.value_counts()

False    83401
True      6217
Name: count, dtype: int64

In [28]:
df.loc[~review_consistency, [
    "name",
    "positive",
    "negative",
    "num_reviews_total",
    "pct_pos_total"
]].head(5)

,name,positive,negative,num_reviews_total,pct_pos_total
0,Counter-Strike 2,7480813,1135108,8632939,86
1,PUBG: BATTLEGROUNDS,1487960,1024436,2513842,59
2,Dota 2,1998462,451338,2452595,81
3,Grand Theft Auto V Legacy,1719950,250012,1803832,87
4,Tom Clancy's Rainbow Six® Siege,1152763,218446,1168020,84


In [29]:
df.loc[~review_consistency, [
    "name",
    "positive",
    "negative",
    "num_reviews_total",
    "pct_pos_total"
]].tail(5)

,name,positive,negative,num_reviews_total,pct_pos_total
89613,Outrun Them,2,0,-1,-1
89614,Lands of Pharaoh: Episode 1,2,0,-1,-1
89615,Player One,2,1,-1,-1
89616,DragonRoad,2,0,-1,-1
89617,Ragtag Adventurers,8,4,-1,-1


In [30]:
review_difference = (
    df["positive"] + df["negative"]
    - df["num_reviews_total"]
)

print("Higher:", (review_difference > 0).sum())
print("Lower:", (review_difference < 0).sum())
print("Equal:", (review_difference == 0).sum())

Higher: 72288
Lower: 11113
Equal: 6217


### We can see that a difference was notice, we will figure why it is happening and fix

To remember: pct = Percentage of Positive Reviews Total

In [20]:
df["pct_pos_total"].value_counts().sort_index()

pct_pos_total
-1      36419
 0         18
 1          1
 2          1
 3          1
        ...  
 96      1189
 97       875
 98       489
 99       119
 100     3044
Name: count, Length: 102, dtype: int64

In [31]:
df["num_reviews_total"].value_counts().sort_index()

num_reviews_total
-1          36419
 10          1687
 11          1696
 12          1440
 13          1349
            ...  
 1168020        1
 1803832        1
 2452595        1
 2513842        1
 8632939        1
Name: count, Length: 5346, dtype: int64

## Review for Data Treatment

During the data consistency check, invalid values (-1) were identified in both pct_pos_total and num_reviews_total.

However, several games with these invalid values still contained valid positive and negative review counts. This suggests that the issue may be related to data collection rather than the absence of user reviews.

To address this limitation, the invalid values were replaced with NaN, and two new variables will be created:

- calculated_total_reviews — Sum of positive and negative reviews.
- calculated_rating — Percentage of positive reviews based on the calculated total.
  
This approach preserves the original information while providing mathematically consistent alternative metrics for popularity and customer satisfaction analysis. However, differences in data collection methods may prevent direct equivalence with the original metrics.

In [32]:
df[["pct_pos_total", "num_reviews_total"]] = df[["pct_pos_total", "num_reviews_total"]].replace(-1, np.nan)

In [33]:
(df[["pct_pos_total", "num_reviews_total"]] == -1).sum()

pct_pos_total        0
num_reviews_total    0
dtype: int64

In [34]:
df.isna().sum()

appid                           0
name                            0
release_date                    0
price                           0
mac                             0
categories                      0
genres                          0
metacritic_score                0
positive                        0
negative                        0
average_playtime_forever        0
median_playtime_forever         0
peak_ccu                        0
pct_pos_total               36419
num_reviews_total           36419
dtype: int64

In [38]:
df[(df["genres"].str.contains("Free To Play", na=False)) &
   (df["price"] > 0)]

,appid,name,release_date,price,mac,categories,genres,metacritic_score,positive,negative,average_playtime_forever,median_playtime_forever,peak_ccu,pct_pos_total,num_reviews_total
2552,2904000,The Spell Brigade,2024-09-16,9.99,False,"['Single-player', 'Multi-player', 'Co-op', 'On...","['Action', 'Casual', 'Indie', 'Free To Play', ...",0,4229,483,1109,580,1787,89.0,4218.0
4091,520720,Dear Esther: Landmark Edition,2017-02-14,9.99,True,"['Single-player', 'Steam Achievements', 'Full ...","['Adventure', 'Casual', 'Indie', 'Free To Play']",0,4969,1705,66,36,10,79.0,2017.0
4179,1999520,CATO: Buttered Cat,2024-09-05,10.99,True,"['Single-player', 'Multi-player', 'Co-op', 'Sh...","['Adventure', 'Casual', 'Indie', 'Free To Play']",0,0,0,0,0,41,98.0,1953.0
5080,1449070,Journey to the West,2023-06-28,19.99,True,"['Single-player', 'Steam Achievements', 'Steam...","['Adventure', 'Indie', 'RPG', 'Strategy', 'Fre...",0,1358,311,1948,1948,30,80.0,1435.0
7417,2609610,CONVRGENCE,2024-04-22,19.99,False,"['Single-player', 'Steam Achievements', 'Track...","['Action', 'Adventure', 'Free To Play', 'Early...",0,702,140,0,0,6,82.0,724.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85135,2224750,B1700,2024-01-13,9.99,False,"['Multi-player', 'PvP', 'Online PvP', 'LAN PvP...","['Action', 'Adventure', 'Casual', 'Indie', 'Ma...",0,4,2,0,0,0,NaN,NaN
85556,2376870,MechCrisis,2024-09-30,1.99,False,"['Single-player', 'Steam Achievements', 'Full ...","['Action', 'Adventure', 'Casual', 'Free To Play']",0,4,1,0,0,0,NaN,NaN
87122,2495510,ModelMaker,2023-12-18,3.99,False,"['Single-player', 'Family Sharing']","['Casual', 'Indie', 'Simulation', 'Free To Play']",0,2,3,0,0,0,NaN,NaN
88267,2515860,What?!,2024-03-14,5.99,False,"['Single-player', 'Family Sharing']","['Adventure', 'Casual', 'Indie', 'RPG', 'Free ...",0,3,0,0,0,0,NaN,NaN
